# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
spaceship = spaceship.dropna(axis=0)

In [4]:
spaceship['Cabin'] = spaceship ['Cabin']. str[0]

In [5]:
spaceship = spaceship.drop(columns = ['PassengerId', 'Name'], axis=1)

In [6]:
cols_non_numeric = ['HomePlanet','CryoSleep','Cabin','VIP','Transported']

X = spaceship.drop(columns=['Destination'])
y = spaceship['Destination']

# Drop rows with missing values
X = X.dropna(subset=cols_non_numeric)
y = y.loc[X.index]

**Perform Train Test Split**

features = spaceship
target = spaceship["Destination"]

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# One-hot encode
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop='first')
ohe.fit(X_train[cols_non_numeric])

X_train_trans_np = ohe.transform(X_train[cols_non_numeric])
X_test_trans_np = ohe.transform(X_test[cols_non_numeric])


In [34]:
# 2.1 Criar DataFrames das colunas codificadas
# Usamos ohe.get_feature_names_out() para obter os nomes das novas colunas
feature_names_out = ohe.get_feature_names_out(cols_non_numeric)

X_train_norm_df = pd.DataFrame(X_train_trans_np, 
                           columns=feature_names_out, 
                           index=X_train.index)

X_test_norm_df = pd.DataFrame(X_test_trans_np, 
                          columns=feature_names_out, 
                          index=X_test.index)

# 2.2 Remover as colunas categóricas originais
X_train_numeric = X_train.drop(columns=cols_non_numeric)
X_test_numeric = X_test.drop(columns=cols_non_numeric)

# CORREÇÃO 2: Usar o DataFrame OHE e não o array NP. Usar nomes finais diferentes.
X_train_final = pd.concat([X_train_numeric, X_train_norm_df], axis=1)
X_test_final = pd.concat([X_test_numeric, X_test_norm_df], axis=1)


In [9]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_test  = pd.get_dummies(X_test, drop_first=True)

# Para garantir que ambas tenham as mesmas colunas
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

Now perform the same as before:
- Feature Scaling
- Feature Selection


In [10]:
normalizer = MinMaxScaler() # x_new = (x - min(x)) / (max(x) -min(x))

In [11]:
normalizer.fit(X_train)

MinMaxScaler()

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state=0)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(5284, 11) (1322, 11) (5284,) (1322,)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

In [13]:
from sklearn.datasets import  fetch_california_housing
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

- Bagging and Pasting

In [14]:
from sklearn.ensemble import BaggingClassifier
bagging_reg = BaggingRegressor(random_state=42) # ou com um regressor base específico

In [37]:
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import BaggingClassifier # MUDAR AQUI
from sklearn.tree import DecisionTreeClassifier # MUDAR AQUI

# 1. Codificar o Target (y_train e y_test)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# 2. Re-instanciar o modelo como Classificador
bagging_clf = BaggingClassifier(DecisionTreeClassifier(max_depth=20),
                                n_estimators=100,
                                max_samples = 1000,
                                random_state=42)

# 3. Treinar com o target codificado
bagging_clf.fit(X_train_final, y_train_encoded)

BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=20),
                  max_samples=1000, n_estimators=100, random_state=42)

In [38]:

y_pred_test_bag = bagging_clf.predict(X_test_final) 

In [39]:
acuracia_bag = accuracy_score(y_test_encoded, y_pred_test_bag)
print(f"Acurácia: {acuracia_bag: .4f}")

Acurácia:  0.7126


In [40]:
from sklearn.metrics import precision_score

# 'weighted' leva em conta o desequilíbrio entre as classes
precisao = precision_score(y_test_encoded, y_pred_test_bag, average='weighted', zero_division=0)
print(f"Precisão (Média): {precisao: .4f}")

Precisão (Média):  0.6556


In [41]:
from sklearn.metrics import recall_score

# 'weighted' leva em conta o desequilíbrio entre as classes
recall = recall_score(y_test_encoded, y_pred_test_bag, average='weighted', zero_division=0)
print(f"Recall (Média): {recall: .4f}")

Recall (Média):  0.7126


In [42]:
from sklearn.metrics import f1_score

# 'weighted' leva em conta o desequilíbrio entre as classes
f1 = f1_score(y_test_encoded, y_pred_test_bag, average='weighted', zero_division=0)
print(f"F1 Score (Média): {f1: .4f}")

F1 Score (Média):  0.6621


In [43]:
from sklearn.metrics import classification_report

print("--- RELATÓRIO DE CLASSIFICAÇÃO ---")
print(classification_report(y_test_encoded, y_pred_test_bag, zero_division=0))

--- RELATÓRIO DE CLASSIFICAÇÃO ---
              precision    recall  f1-score   support

           0       0.47      0.20      0.28       260
           1       0.30      0.09      0.14       109
           2       0.75      0.92      0.83       953

    accuracy                           0.71      1322
   macro avg       0.51      0.41      0.42      1322
weighted avg       0.66      0.71      0.66      1322



- Random Forests

In [45]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
import numpy as np 

In [56]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100,
                             max_depth=20)

In [57]:
forest.fit(X_train_final, y_train_encoded) 

RandomForestClassifier(max_depth=20)

In [58]:
y_pred_test_rf = forest.predict(X_test_final) 

In [59]:
print("\n--- AVALIAÇÃO DO RANDOM FOREST CLASSIFIER ---\n")


--- AVALIAÇÃO DO RANDOM FOREST CLASSIFIER ---



In [60]:
# 2.2 Métricas de Classificação

# Acurácia
accuracy_rf = accuracy_score(y_test_encoded, y_pred_test_rf)
print(f"Acurácia (Accuracy): {accuracy_rf: .4f}")

# F1 Score
f1_rf = f1_score(y_test_encoded, y_pred_test_rf, average='weighted', zero_division=0)
print(f"F1 Score (Média Ponderada): {f1_rf: .4f}")

# Score Nativo (Acurácia)
rf_score = forest.score(X_test_final, y_test_encoded)
print(f"Score Nativo do Modelo: {rf_score: .4f}")

# Relatório Detalhado
print("\n--- Relatório Detalhado por Classe (RANDOM FOREST) ---\n")
print(classification_report(y_test_encoded, y_pred_test_rf, zero_division=0))

Acurácia (Accuracy):  0.6936
F1 Score (Média Ponderada):  0.6489
Score Nativo do Modelo:  0.6936

--- Relatório Detalhado por Classe (RANDOM FOREST) ---

              precision    recall  f1-score   support

           0       0.37      0.19      0.25       260
           1       0.32      0.11      0.16       109
           2       0.74      0.90      0.81       953

    accuracy                           0.69      1322
   macro avg       0.48      0.40      0.41      1322
weighted avg       0.63      0.69      0.65      1322



- Gradient Boosting

In [63]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_clf = AdaBoostClassifier(
    # Base Estimator (Modelo Base): Deve ser um Classificador
    DecisionTreeClassifier(max_depth=20),
    
    # n_estimators: Número de modelos (árvores) que serão construídos
    n_estimators=100,
    
    # random_state (Boa Prática): Garante a reprodutibilidade
    random_state=42 
)

In [66]:
# Treinamento (Use seu X_train limpo e o target codificado)
ada_clf.fit(X_train_final, y_train_encoded) 

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=20),
                   n_estimators=100, random_state=42)

In [67]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
# Lembre-se que X_test_final é o seu X_test limpo (substitua pelo seu nome, se for X_test_norm_df)

# 1. Gerar as previsões do AdaBoost Classifier
y_pred_test_ada = ada_clf.predict(X_test_final)

print("--- Métricas do AdaBoost Classifier ---")

# 2. Acurácia (Equivalente mais próximo do 'score' nativo)
accuracy_ada = accuracy_score(y_test_encoded, y_pred_test_ada)
print(f"Acurácia (Accuracy): {accuracy_ada: .4f}")

# 3. F1 Score (Métrica de balanceamento, mais informativa que a Acurácia)
# 'weighted' é essencial para multi-classes
f1_ada = f1_score(y_test_encoded, y_pred_test_ada, average='weighted', zero_division=0)
print(f"F1 Score (Média Ponderada): {f1_ada: .4f}")

# 4. Score Nativo (Retorna a Acurácia para Classificadores)
ada_score = ada_clf.score(X_test_final, y_test_encoded)
print(f"Score Nativo do Modelo: {ada_score: .4f}")

print("\n--- Relatório Detalhado por Classe ---\n")
# 5. Relatório Completo (Inclui Precision e Recall por classe)
print(classification_report(y_test_encoded, y_pred_test_ada, zero_division=0))

--- Métricas do AdaBoost Classifier ---
Acurácia (Accuracy):  0.6513
F1 Score (Média Ponderada):  0.6266
Score Nativo do Modelo:  0.6513

--- Relatório Detalhado por Classe ---

              precision    recall  f1-score   support

           0       0.30      0.22      0.25       260
           1       0.21      0.13      0.16       109
           2       0.74      0.83      0.78       953

    accuracy                           0.65      1322
   macro avg       0.42      0.39      0.40      1322
weighted avg       0.61      0.65      0.63      1322



- Adaptive Boosting

In [69]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(
    # max_depth: A profundidade das árvores (o GB funciona melhor com árvores rasas)
    max_depth=20,  
    
    # n_estimators: O número de estágios de aumento (ou o número de árvores)
    n_estimators=100,
    
    # Adicionando learning_rate (Boa prática para GB)
    learning_rate=0.1, 
    
    # Garante a reprodutibilidade
    random_state=42 
)

In [71]:
gb_clf.fit(X_train_final, y_train_encoded) 

GradientBoostingClassifier(max_depth=20, random_state=42)

In [73]:
y_pred_test_gb = gb_clf.predict(X_test_final)

In [74]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd

# =================================================================================
# 1. GERAR PREVISÕES PARA TODOS OS MODELOS TREINADOS
# =================================================================================

# Assumindo:
# - X_test_final é o seu DataFrame de teste limpo.
# - y_test_encoded é o seu target de teste codificado.

print("Gerando previsões para os 3 modelos...")

# Previsões do Bagging Classifier
y_pred_test_bag = bagging_clf.predict(X_test_final) 

# Previsões do Random Forest Classifier
y_pred_test_rf = forest.predict(X_test_final) 

# Previsões do Gradient Boosting Classifier
y_pred_test_gb = gb_clf.predict(X_test_final) 

print("Previsões concluídas.")

# =================================================================================
# 2. CALCULAR MÉTRICAS E CONSOLIDAR EM UM DATAFRAME
# =================================================================================

modelos = ['Bagging Classifier', 'Random Forest Classifier', 'Gradient Boosting Classifier']
previsoes = [y_pred_test_bag, y_pred_test_rf, y_pred_test_gb]
resultados = []

for nome, predicao in zip(modelos, previsoes):
    # Acurácia
    accuracy = accuracy_score(y_test_encoded, predicao)
    
    # F1 Score (Métrica mais robusta para multi-classes desbalanceadas)
    f1 = f1_score(y_test_encoded, predicao, average='weighted', zero_division=0)
    
    # Adicionar os resultados à lista
    resultados.append({
        'Modelo': nome,
        'Acurácia': f'{accuracy:.4f}',
        'F1 Score (Weighted)': f'{f1:.4f}'
    })

# Criar o DataFrame de resultados
df_resultados = pd.DataFrame(resultados)

print("\n" + "="*70)
print("             TABELA DE RESULTADOS CONSOLIDADOS")
print("="*70)
print(df_resultados.to_string(index=False))
print("="*70)


# =================================================================================
# 3. RELATÓRIOS DETALHADOS (OPCIONAL, mas útil)
# =================================================================================

print("\n--- RELATÓRIO DETALHADO DO MELHOR MODELO (EX: RANDOM FOREST) ---\n")
# Você pode escolher o modelo com melhor F1 Score para detalhar
print(classification_report(y_test_encoded, y_pred_test_rf, zero_division=0))

# print("\n--- MATRIZ DE CONF

Gerando previsões para os 3 modelos...
Previsões concluídas.

             TABELA DE RESULTADOS CONSOLIDADOS
                      Modelo Acurácia F1 Score (Weighted)
          Bagging Classifier   0.7126              0.6621
    Random Forest Classifier   0.6936              0.6489
Gradient Boosting Classifier   0.6846              0.6448

--- RELATÓRIO DETALHADO DO MELHOR MODELO (EX: RANDOM FOREST) ---

              precision    recall  f1-score   support

           0       0.37      0.19      0.25       260
           1       0.32      0.11      0.16       109
           2       0.74      0.90      0.81       953

    accuracy                           0.69      1322
   macro avg       0.48      0.40      0.41      1322
weighted avg       0.63      0.69      0.65      1322



Which model is the best and why?

F1 Score (Weighted)
          Bagging Classifier   0.7126              0.6621
    Random Forest Classifier   0.6936              0.6489
Gradient Boosting Classifier   0.6846              0.6448
In our classification project to predict the Destination, we compared three ensemble models: Bagging, Random Forest, and AdaBoost. After careful data preparation involving One-Hot Encoding and target encoding, the models were evaluated using the F1 Score and Accuracy. The Random Forest Classifier is typically the most stable choice due to its variance reduction, but the model with the highest F1 Score in the final consolidated results should be selected as the best performer.